#analise do comportamento de clientes#

In [0]:
## ADICIONANDO AS SOMATÓRIAS DE ELOGIOS PARA REVIEWS DE RESTAURANTES

from pyspark.sql.functions import col, ntile, when, count, sum
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_ing"

# Carrega tabelas silver
df_user = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_users_filtered")
df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review_filtered")

# ========== ADICIONANDO CAMPO review_cool ==========

print("ADICIONANDO campo review_cool")
print("="*60)

# Soma os valores da coluna "cool" por user_id
df_review_cool_count = df_review.groupBy("user_id").agg(
    sum("cool").alias("review_cool")
)

df_user = df_user.join(df_review_cool_count, "user_id", "left")

# ========== ADICIONANDO CAMPO review_funny ==========

print("ADICIONANDO campo review_funny")
print("="*60)

# Soma os valores da coluna "cool" por user_id
df_review_funny_count = df_review.groupBy("user_id").agg(
    sum("funny").alias("review_funny")
)
df_user = df_user.join(df_review_funny_count, "user_id", "left")

# ========== ADICIONANDO CAMPO review_useful ==========

print("ADICIONANDO campo review_useful")
print("="*60)

# Soma os valores da coluna "useful" por user_id
df_review_useful_count = df_review.groupBy("user_id").agg(
    sum("useful").alias("review_useful")
)
df_user = df_user.join(df_review_useful_count, "user_id", "left")

display(df_user.limit(20))

In [0]:
## NOVO SCORE CALCULADO

from pyspark.sql.functions import col

# Configuração
CATALOG = "workspace"
SCHEMA_GOLD = "yelp_ing"

df_user = df_user.withColumn(
    "score",
    col("fans") +
    col("review_cool") +
    col("review_funny") +
    col("review_useful") +
    col("review_food_count")*10  # peso maior para review_food_count
)

df_user.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.gold_user_behavior")

display(spark.table(f"{CATALOG}.{SCHEMA_GOLD}.gold_user_behavior").limit(20))


In [0]:
## CRIANDO TABELA COM SCORE DE USUÁRIOS POR LOCALIZAÇÃO

from pyspark.sql.functions import col, count, sum as spark_sum, lit

# Configuração
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_ing"
SCHEMA_GOLD = "yelp_ing"

print("Carregando tabelas...")
print("="*60)

# Carrega tabelas necessárias
df_user = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_users_filtered").limit(100)
df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review_filtered")
df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business_filtered")

print("Agregando dados por user_id + city + state...")
print("="*60)

# JOIN: user -> review -> business (para pegar city/state)
df_user_location = (
    df_review
    .join(df_business.select("business_id", "city", "state"), "business_id", "inner")
    .join(df_user.select("user_id", "name", "fans", "average_stars", "yelping_since"), "user_id", "inner")
    .filter((col("city").isNotNull()) & (col("state").isNotNull()))
)

# Agregação por user_id + city + state
df_user_location_agg = (
    df_user_location
    .groupBy("user_id", "name", "city", "state", "fans", "average_stars", "yelping_since")
    .agg(
        count("review_id").alias("review_count_na_localidade"),
        spark_sum("cool").alias("review_cool_na_localidade"),
        spark_sum("funny").alias("review_funny_na_localidade"),
        spark_sum("useful").alias("review_useful_na_localidade")
    )
)

# Calcula score_na_localidade
df_user_location_agg = df_user_location_agg.withColumn(
    "score_na_localidade",
    col("fans") +
    col("review_cool_na_localidade") +
    col("review_funny_na_localidade") +
    col("review_useful_na_localidade") +
    (col("review_count_na_localidade") * 10)  # peso maior para reviews
)
###################

print("Agregando dados por user_id + state...")
print("="*60)

# Agregação por user_id + state
df_user_state_agg = (
    df_user_location
    .groupBy("user_id", "name", "state", "fans", "average_stars", "yelping_since")
    .agg(
        count("review_id").alias("review_count_no_estado"),
        spark_sum("cool").alias("review_cool_no_estado"),
        spark_sum("funny").alias("review_funny_no_estado"),
        spark_sum("useful").alias("review_useful_no_estado")
    )
)

# Calcula score_no_estado
df_user_state_agg = df_user_state_agg.withColumn(
    "score_no_estado",
    col("fans") +
    col("review_cool_no_estado") +
    col("review_funny_no_estado") +
    col("review_useful_no_estado") +
    (col("review_count_no_estado") * 10)  # peso maior para reviews
)

df_final = df_user_location_agg.join(
    df_user_state_agg.select("user_id", col("score_no_estado").alias("score_no_estado")),
    "user_id",
    "left"
)

#display(df_final.orderBy(col("score_no_estado").desc()).limit(20))

######################


#JOIN com gold_user_behavior para trazer score_total
df_gold_user = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.gold_user_behavior")

df_final = df_final.join(
    df_gold_user.select("user_id", col("score").alias("score_total")),
    "user_id",
    "left"
)


print("Salvando tabela gold_user_location_behavior...")
print("="*60)

df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.gold_user_location_behavior"
)

print("✅ Tabela criada com sucesso!")
print(f"Total de registros: {df_final.count()}")
print("\nAmostra dos dados:")
display(df_final.orderBy(col("score_na_localidade").desc()).limit(20))